# CodeCompass on the SCIS GPU cluster

Runs the CodeCompass generator (Qwen2.5-Coder-14B through llama.cpp) on a GPU notebook, then answers questions about a git repository.

**Status (2026-11-03):** works end to end on an RTX 3090.
- Indexing Flask takes a few minutes on the GPU.
- With the GPU re-ranker, a question takes about 0.5 to 5 seconds. On the CPU it took 160 to 230 seconds.
- The model calls tools (`open_file`, `git_log`, `git_blame`) when a question passes the evidence gate.
- Most answers come back flagged (`G`), so check every source by hand.

**Known issues**
- The prebuilt llama.cpp binary needs a newer glibc than the cluster has. Build from source (section 2).
- The evidence gate (`min_top_score` = 0.02) blocks vague questions before the model runs. Name a specific file or function in each question.
- The embedder's remote code (`jinaai/jina-bert-v2-qk-post-norm`) is not pinned to a revision. Hugging Face warns about it when it loads.
- Indexing prints a tokenizer warning (14103 > 8192 tokens). Not yet checked whether it affects the index.

**After a kernel restart** the notebook loses all variables and the server stops. Run section 0, then section 4 (server). Skip sections 1 to 3 unless you're setting up a new machine.

**Account and data rules** (from scis_tech@smu.edu.sg)
- This is your own cluster account. Do not share it. Connect to GlobalProtect VPN first, and only one VPN at a time.
- Nothing on the cluster is backed up. Keep the project zip and anything you need in a copy outside the cluster.
- Files in the scratch folder older than 30 days are deleted automatically.
- Home directory data is deleted one month after the account expires and cannot be recovered.
- The model (about 9 GB), the llama.cpp build and the index live in your home folder. Re-downloading and rebuilding takes about an hour, so keep a backup of anything you can't afford to lose.
- Do not put passwords or tokens in this notebook.


## 0. Paths and config (run after every kernel restart)

In [ ]:
import json
import os
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path

HOME = Path.home()
PROJECT = HOME / "CodeCompass-project"
REPO = HOME / "repos/flask"  # a git repo you may index; never fastapi/fastapi (reserved for evaluation)
MODEL = HOME / ".cache/codecompass/models/qwen2.5-coder-14b-instruct-q4_k_m.gguf"
SERVER_BIN = HOME / "llama.cpp-src/build/bin/llama-server"
USER_CFG = HOME / ".config/codecompass/codecompass.toml"

os.environ["GENERATOR_PROFILE"] = "dev"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("project:", PROJECT)
print("repo:", REPO)
print("model file present:", MODEL.exists(), "| server binary present:", SERVER_BIN.exists())

In [ ]:
from codecompass.online import ask
from codecompass.online.config import load_online_config_layers

USER_CFG.parent.mkdir(parents=True, exist_ok=True)
if not USER_CFG.exists():
    USER_CFG.write_text(
        '[embedder]\ndevice = "cuda"\n\n'
        '[reranker]\ndevice = "cuda"\ndtype = "float16"\n'
    )

# Layered load: your user file first, then the repo file.
# The loader ignores your user file if you load only the repo file, so always use this function.
config = load_online_config_layers([USER_CFG, PROJECT / "codecompass.toml"])
print(config.reranker)

## 1. One-time setup: check the GPU and unzip the project

Upload `CodeCompass-project.zip` to your home folder first. Leave out `.venv`, `__pycache__`, `.git`, `runs` and `idx` when you zip it.

In [ ]:
import subprocess

import torch

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout if smi.returncode == 0 else "nvidia-smi not available")
print("PyTorch CUDA available:", torch.cuda.is_available())

In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path

ZIP = Path.home() / "CodeCompass-project.zip"
PROJECT = Path.home() / "CodeCompass-project"
if not PROJECT.exists():
    if not zipfile.is_zipfile(ZIP):
        raise RuntimeError(f"{ZIP} is not a valid zip; re-upload it")
    zipfile.ZipFile(ZIP).extractall(Path.home())

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    check=True,
)

## 2. One-time setup: build llama-server from source

The prebuilt binaries need glibc 2.38, and the cluster has 2.34. This builds the same pinned release (b11960) with the cluster's CUDA 12.6. It takes 10 to 30 minutes. `cmake` is installed only in this notebook's environment.

In [ ]:
%pip install -q cmake

In [ ]:
%%bash
set -e
cd ~
if [ ! -d llama.cpp-src ]; then
  git clone --depth 1 --branch b11960 https://github.com/ggml-org/llama.cpp llama.cpp-src
fi
cd llama.cpp-src
cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
cmake --build build --config Release -j "$(nproc)" --target llama-server
ls -l build/bin/llama-server

## 3. One-time setup: download the model

About 9 GB. The SHA-256 must match `codecompass.toml`. If it doesn't, the file is deleted and the cell raises an error.

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

MODEL = Path.home() / ".cache/codecompass/models/qwen2.5-coder-14b-instruct-q4_k_m.gguf"
EXPECTED = "c1e659736d89ac1065fb495330fb824d94001974a4bfa78e7270e43476a8d940"

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not MODEL.exists():
    MODEL.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct-GGUF/resolve/main/"
        "qwen2.5-coder-14b-instruct-q4_k_m.gguf",
        MODEL,
    )
if sha256_of(MODEL) != EXPECTED:
    MODEL.unlink()
    raise RuntimeError("checksum mismatch: file deleted, run this cell again")
print("model ok:", MODEL)

## 4. Start the generator server (after every restart)

If an old kernel still has a server on port 8080, this cell exits early. Stop the old one first (section 10 in that kernel, or restart that kernel).

In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path

env = dict(os.environ)
env["LD_LIBRARY_PATH"] = os.pathsep.join(
    p for p in [str(SERVER_BIN.parent), "/opt/apps/software/CUDA/12.6.0/lib64", os.environ.get("LD_LIBRARY_PATH", "")] if p
)
cmd = [str(SERVER_BIN), "-m", str(MODEL), "--host", "127.0.0.1", "--port", "8080",
       "-c", "32768", "--jinja", "-ngl", "99"]

log = open(Path.home() / "llama-server.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

for _ in range(120):
    if server.poll() is not None:
        print("server exited early with code", server.returncode, "; see ~/llama-server.log")
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2) as r:
            print(r.read().decode())
            break
    except OSError:
        time.sleep(5)

In [ ]:
import json
import urllib.request

req = urllib.request.Request(
    "http://127.0.0.1:8080/v1/chat/completions",
    data=json.dumps({"messages": [{"role": "user", "content": "Say hi"}], "max_tokens": 16}).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=120) as r:
    print(json.loads(r.read())["choices"][0]["message"]["content"])

## 5. Clone the repo to query and build the index

Indexing is incremental, so running this again on the same commit finishes quickly. The first run downloads the embedder (about 600 MB) and takes a few minutes for Flask.

The output is captured, so the cell shows nothing until indexing ends. Use section 9 to watch progress.

In [ ]:
import subprocess
import sys

if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/pallets/flask", str(REPO)], check=True)

idx = subprocess.run(
    [sys.executable, "-m", "codecompass.offline.cli", "index", str(REPO), "--embedder", "code"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(idx.stdout[-2000:], idx.stderr[-2000:])

st = subprocess.run(
    [sys.executable, "-m", "codecompass.offline.cli", "status", str(REPO)],
    cwd=PROJECT, capture_output=True, text=True,
)
print(st.stdout, st.stderr)

## 6. Ask one question

Name a specific file or function. Vague questions can be blocked by the evidence gate before the model runs. Check every citation yourself: `F` means the cited code was in the evidence, not that the claim is true.

In [ ]:
question = "Where is configuration loaded?"
result = ask(str(REPO), question, config=config)

print("Outcome:", result.citation_outcome)
print()
print(result.answer)
print()
print("Audit log:", result.audit_log["log_file"])

## 7. Ask several questions and time them

In [ ]:
questions = [
    "Where is configuration loaded?",
    "Where is the request context created?",
]

for q in questions:
    start = time.perf_counter()
    r = ask(str(REPO), q, config=config)
    print(f"{time.perf_counter() - start:6.1f}s  {r.citation_outcome}  {q}")

## 8. Test tool calling

The first check reads the parser. The second asks questions that name files, and counts the tool calls the model makes. Runs that the gate blocks never reach the model, so they say nothing about tool use.

Your teammates' earlier result (8 of 8 gated runs made a tool call) came from this cell.

In [ ]:
from codecompass.online.agent.tool_parser import malformed_blocks, parse_tool_calls

print(parse_tool_calls('<tool>git_log(path="README.md", n=2)</tool>'))
print(malformed_blocks('<tool>git_log(path=)</tool>'))

In [ ]:
TOOL_QUESTIONS = [
    "Use open_file to show lines 1-30 of src/flask/app.py and explain what it imports.",
    "Use open_file to show lines 1-20 of src/flask/helpers.py and say what the module is for.",
    "Use git_log on src/flask/app.py and tell me the most recent commit that changed it.",
    "Use git_blame on src/flask/app.py lines 1-20 and say who last changed the imports.",
    "Use git_log on README.md and tell me the most recent commit that changed it.",
]
RUNS_PER_QUESTION = 2

summary = Counter()
gated = 0
for q in TOOL_QUESTIONS:
    for _ in range(RUNS_PER_QUESTION):
        r = ask(str(REPO), q, config=config)
        log = json.loads(Path(r.audit_log["log_file"]).read_text())
        if log["generator_calls"] == 0:
            gated += 1
        for c in log["tool_calls"]:
            summary[c.get("tool") or "malformed"] += 1
        print(f"{r.citation_outcome}  tools={[c.get('tool') for c in log['tool_calls']]}  "
              f"gate={log['evidence_gate']['passed']}  {q}")

print("\nTool requests:", dict(summary))
print("Blocked by the evidence gate before the model ran:", gated)

## 9. Check progress from a second notebook

The timing and tool cells print only when each question ends. Run these in a second notebook while a cell is busy.

In [ ]:
from pathlib import Path

lines = (Path.home() / "llama-server.log").read_text(errors="replace").splitlines()
print("started:", sum("processing task" in l for l in lines),
      "| finished:", sum("stop processing" in l for l in lines))
print("\n".join(lines[-3:]))

In [ ]:
import subprocess

print(subprocess.run(
    ["bash", "-c", "ps -eo pid,pcpu,etime,args --sort=-pcpu | grep -i ipykernel | grep -v grep | head -5"],
    capture_output=True, text=True,
).stdout)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
import subprocess

root = "${XDG_DATA_HOME:-$HOME/.local/share}/codecompass"
print(subprocess.run(["bash", "-c", f"du -sh {root}; find {root} -maxdepth 3 | head -30"],
                     capture_output=True, text=True).stdout)

## 10. Stop the server

Stop the server before you end the session so the GPU is free for others.

In [ ]:
server.terminate()
server.wait(timeout=30)
print("server stopped:", server.poll())